# **Introduction (Assignment 1, Deadline 23/10/2026 at 08:00 AM)**

Students write text here, explaining:
 - What it means to preprocess data and what is a preprocessing pipeline
 - What are the main artefacts that can be found in the TMS-EEG signal and their causes
 - **Optional**: a personal comment or opinion on the problems posed by preprocessing, based on what you heard in class
   
Data preprocessing is a set of operations aimed at separating the signal, that is the fraction of scalp voltage generated by the cortex in response to the TMS pulse, from the noise, that is a signal generated by other agents such as muscles and electronic instrumentation and that is not interesting in respect to the analysis. A preprocessing pipeline consists of a series of multiple transformations done in a precise sequence in order to separate signal from noise. Nowadays there is not a standard preprocessing pipeline; however, a broad and schematic classification can be made based upon the philosophical approach underlying the preprocessing pipeline. Conservationist approaches try to preserve as much information as possible, taking the risk of having more noisy data; instead, interventionist approaches act heavily on the signal, prioritizing the minimization of noise at the risk of losing some significant information. The main artefacts that can be found in the raw TMS-EEG signal are eyeblinks artefacts, muscles artefacts and TMS pulse artefacts. Eyes artefacts are caused by ocular movements, they are much larger than the rest of the signal and have a limited temporal duration, they are therefore relatively easily localized and are more visible in the signal registered in the frontal channels. Muscle artefacts have high amplitude, high frequency and are present in several channels; they generate a signal that is noticeably larger than the cortical one and cancels the latter. A third kind of artefact is the one generated by the TMS pulse at the time of the stimulation. It is a very strong signal that completely overrides the cortical signal and has to be eliminated in order to make the data usable; this part of the preprocessing pipeline is done using interpolation.
The subjectivity in the choice of the epistemological approach to the preprocessing pipeline, that can be more conservationist or more interventionist, might be a problem in terms of the reproducibility of scientific findings. I find particularly interesting how the physiological needs determine the methodological approach and impose certain trade-offs, however without a shared standardization the prospective of a shared knowledge might be more difficult. For what it concerns reproducibility, another issue is the need of transparency in every step of the pipeline when sharing a scientific research.


In [2]:
from pathlib import Path                            # to build a bridge between Python and the filesystem 

import numpy as np                                  # to perform array operations
import matplotlib                                   # this and the following to draw plots
import matplotlib.pyplot as plt
matplotlib.use('Qt5Agg')                            # to make plots interactive

import mne                                          # to read and manipulate EEG data

from scripts import utils                           # custom functions to shorten the code in this notebook

# **1. Basic Preprocessing (Assignment 1, Deadline 23/10/2026 at 08:00 AM)**

Students write text here, listing the operations that we categorise as "basic preprocessing".

If you cannot remember what the steps are, look at the code in the cells below and you will recall.

Basic preprocessing can be intended as a small set of essential operations that can be summarized as follows:
 - Loading the data and making them available to the software
 - Inspecting data
 - Adjusting data
 - Interpolating
 - High-pass filtering
 

## **1.1. Data loading and availability**

**Optional but welcome:** students write here any considerations about this step.

In the first part of this step there is the definition of two variables: data_dir is the name chosen for the directory where the data are stored and subject_and_condition is a defined variable that in this case identifies the subject 02 in condition 1.
In the string with the definition of file_of_interest we can see the suffix .vhdr. In the BrainVision format, that is the one used in this study, information is spread across 3 files:
 1) .eeg = eeg time series
 2) .vhdr = metadata, they give context to the data
 3) .vmrk = information about event marker

In the second block of code we can see MNE, an open-source packet that is the gold standard for EEG analysis.
In this step data are imported and made available to the software for subsequent operations.


In [3]:
data_dir = Path("data")
subject_and_condition = "S02C1_M1"

file_of_interest = list(data_dir.glob(pattern=f"{subject_and_condition}.vhdr"))
print(file_of_interest, "|", type(file_of_interest), "|", file_of_interest[0])

[WindowsPath('data/S02C1_M1.vhdr')] | <class 'list'> | data\S02C1_M1.vhdr


In [4]:
eeg_data = mne.io.read_raw_brainvision(vhdr_fname=file_of_interest[0])

Extracting parameters from data\S02C1_M1.vhdr...
Setting channel info structure...


## **1.2. Channels dropping (adjusting data)**

**Compulsory:** students write here the functional significance of this step (that is, why we do it).

This step consists in the elimination of unwanted channels. The discrimination between wanted and unwanted channels is done leveraging their denomination. Central channels end with z (zenit), channels on the right with even numbers and those on the left with odd numbers. The numbers are assigned in an ascending order from medial to lateral regions. The subsequent block of code defines as channels to be dropped those whose last character of the name is not a number and is not a "z". Here the output states that the channels EOG (ElectroOculoGram - ocular activity) and FDI (First Dorsal Interosseous muscle) must be dropped, that is because they are unwanted channels they are not useful to the data analysis since they do not register signal of interest. 

In [5]:
channels_to_drop = []

for channel_name in eeg_data.ch_names:
    try:
        int(channel_name[-1])
    except ValueError:
        if channel_name[-1] != "z": 
            channels_to_drop.append(channel_name)
print(f"Channels to drop: {channels_to_drop}")

eeg_data = eeg_data.drop_channels(ch_names=channels_to_drop)

Channels to drop: ['EOG', 'FDI']


## **1.3. Data inspection**

**Compulsory:** students write here the functional significance of this step (that is, why we do it).

Inspecting the data provides information such as their shape, content and aquisition parameters. The grid below shows information divided in four sections: general (contains metadata), acquisition, channels and filters. The "acquisition" and "channels" section are particularly relevant for data inspection since they contain information about respectively the temporal and spatial dimension of the data. In the "acquisition" section are indicated the duration of the recording (hh: mm :ss), the sampling frequency, that is how many samples per second are saved by the software  (measured in Hz), and the time points. Regarding the latter, EEG data are structurally organized in a matrix in which each element is defined in a time point measured in array units. It is worth noting that all the acquisition parameters have units of measurment that have the concept of time in common, thus making possible the application of formulas to convert between them, mapping between the "real world time" and the data structure.
In order to investigate the spatial dimension, data are standardized using a built-in MNE's easycap-M1 montage, that is compatible with the physical cap used in the study from which the present data are taken. Doing so makes possible the analysis at the level of the channels which is essential for source reconstruction and topoplots.

In [6]:
easycap_m1_montage = mne.channels.make_standard_montage(kind="easycap-M1")
easycap_m1_montage.plot()
eeg_data.set_montage(montage=easycap_m1_montage,
                     on_missing="ignore")

<RawBrainVision | S02C1_M1.eeg, 70 x 3307600 (661.5 s), ~85 KiB, data not loaded>

**Compulsory:** here, students describe and comment the plot generated by the code below. In particular, I would like to read:

- What each row represents
- What are the $x$ and $y$ of each row, and their units of measurement
- A description of the artefacts you see, and their likely causes

The code below generates a plot that is a visual rapresentation of the data matrix. Each row of the plot represents a channel. On the x axis there is time, measured in seconds (s). The y axis of each row is the signal intensity, measured in microvolts (μV); the amplitude visualization span is 40μV peak-to-peak.
The main artefacts visible on the plot are:
1) muscle artefacts, signals with high amplitude, high frequency and ubiquitous; caused by muscular activity
2) eyeblinks artefacts, relatively smaller than muscle artefacts but still larger than cortical signal, more present in the frontal area; caused by ocular activity
3) TMS pulse artefact, a high frequency signal localized at the time of stimuli presentations; caused by the electric current induced by the electromagnetic pulse.


In [7]:
eeg_data.plot()

Using matplotlib as 2D backend.
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`


<MNEBrowseFigure size 1920x986 with 4 Axes>

Channels marked as bad:
none


**Compulsory:** here, students define event markers and explain why they are important for EEG data analysis
**Optional:** any comments that go beyond a mere definition are welcome. For example, you could make comments about the structure of event arrays and what it tells us about the nature of events

Event markers are contained in the .vmrk file and provide information about the ordinal number, type, name, time of occurrence and duration of a stimulus, and also information about the channels involved. Time of occurrence and duration of a stimulus are not measured in seconds but in data points and allow the precise localization of the stimulus in the EEG data matrix. These markers are important for the TMS-EEG signal analysis since they provide the localization of the TMS pulse in the array and therefore of the TMS pulse artefact. This artefact is in fact so strong that it overrides the relevant signal and it needs to be eliminated from the data in order to do the analysis. The following block of code uses MNE to extract marker information from the eeg data structures and translates them into a Python-friendly format. 





In [8]:
events_from_annotations, events_dict = mne.events_from_annotations(raw=eeg_data)
events_from_annotations = events_from_annotations[2:]

Used Annotations descriptions: ['New Segment/', 'Stimulus/S 54', 'Stimulus/S255']


## **1.4. Interpolation**

**Compulsory:** students write here the functional significance of this step (that is, why we do it).

Interpolation is a mathematical technique that estimates the missing values of a function based on the preceding and following values of such function. As mentioned above, the TMS pulse artefacts heavily affect the data and they need to be removed. Doing so will result in loss of the cortical signal at the time of the pulse. Interpolation allows for the estimation of this lost information. Before interpolation, an essential step is segmenting the data, that is dividing the EEG trace into epochs centered on the event (the TMS pulse). 

In [9]:
pre_interpolation_epochs = mne.Epochs(raw=eeg_data,
                                      events=events_from_annotations,
                                      tmin=-1.1,
                                      tmax=0.5,
                                      baseline=None)
pre_interpolation_epochs_tep = pre_interpolation_epochs.average()
pre_interpolation_epochs_tep.plot();

Not setting metadata
200 matching events found
No baseline correction applied
0 projection items activated


In [10]:
post_interpolation_eeg = mne.io.read_raw(fname="data/post_2_5_interpolation_eeg.fif",
                                         preload=True)
post_interpolation_epochs = mne.Epochs(raw=post_interpolation_eeg,
                                       events=events_from_annotations,
                                       tmin=-1.1,
                                       tmax=0.5,
                                       baseline=None)
post_interpolation_tep = post_interpolation_epochs.average()
post_interpolation_tep.plot();

Opening raw data file data/post_2_5_interpolation_eeg.fif...
    Range : 0 ... 3307599 =      0.000 ...   661.520 secs
Ready.
Reading 0 ... 3307599  =      0.000 ...   661.520 secs...
Not setting metadata
200 matching events found
No baseline correction applied
0 projection items activated


## **1.5. High-pass filtering**

**Compulsory:** students write here the functional significance of this step 

EEG data consist of various oscillations at different frequencies. Depending on the goal of the study, some frequencies might not be wanted and there is the need to eliminate them. Filters are methodological tools that are used to study waves at specific frequencies of interest or above or below certain frequencies. Some of the main filters used in data analysis are:
1) high-pass filter, that filters out frequencies below a certain value
2) low-pass filter, that filters out frequencies above a certain value
3) band-pass , that filters out frequencies outside a certain range
4) notch filter, a band stop filter that weakens frequencies inside a certain range (used at 50Hz in Europe and at 60Hz in the United States to eliminate the artifacts generated by the electrical current)
The following block of code applies a high-pass filter at 0.1 Hz to the data, weakening the frequencies lower than 0.1Hz.

Filters work based on the mathematical operation of convolution.





In [11]:
post_filtering_eeg = mne.filter.filter_data(data=post_interpolation_eeg.get_data(),
                                            sfreq=post_interpolation_eeg.info["sfreq"],
                                            l_freq=0.1,
                                            h_freq=None,
                                            method="iir",
                                            iir_params=None,
                                            copy=True,
                                            phase="zero")
post_filtering_eeg = mne.io.RawArray(data=post_filtering_eeg,
                                     info=post_interpolation_eeg.info)
post_filtering_epochs = mne.Epochs(raw=post_filtering_eeg,
                                   events=events_from_annotations,
                                   tmin=-1.1,
                                   tmax=0.5,
                                   baseline=None)
post_filtering_tep = post_filtering_epochs.average()
post_filtering_tep.plot();

Setting up high-pass filter at 0.1 Hz

IIR filter parameters
---------------------
Butterworth highpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 8 (effective, after forward-backward)
- Cutoff at 0.10 Hz: -6.02 dB

Creating RawArray with float64 data, n_channels=70, n_times=3307600
    Range : 0 ... 3307599 =      0.000 ...   661.520 secs
Ready.
Not setting metadata
200 matching events found
No baseline correction applied
0 projection items activated
